# 12 — FINAL TFLite Export + Deployment Latency Audit

## NO TRAINING

This notebook performs the deployment-lock stage for the revised paper.

It will:

1. Audit existing TFLite and latency artifacts already in the Cataract Drive.
2. Resolve the **final MobileNetV2 frozen baseline** and **final MobileNetV2 last-block fine-tuned checkpoint**.
3. Export both to standard float32 TFLite.
4. Record `.keras` and `.tflite` sizes.
5. Verify Keras ↔ TFLite prediction parity on real held-out images.
6. Benchmark controlled TFLite inference latency on 100 real test images.
7. Report mean, SD, median, P95, minimum, maximum, and throughput.
8. Combine accuracy + deployment metrics into one final deployment comparison.
9. Recommend the deployment checkpoint without changing the fair five-model benchmark.

### Important scientific separation

- **Frozen MobileNetV2** remains part of the fair five-model benchmark.
- **Last-block MobileNetV2** is a separate fine-tuning sensitivity/deployment candidate.
- Selecting the fine-tuned checkpoint for deployment does **not** change the frozen five-model benchmark.
- No training occurs here.

In [ ]:
# ============================================================
# CELL 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import os
import re
import time
import shutil
import platform
import subprocess
import sys

import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT = Path('/content/drive/MyDrive/Cataract')
FINAL_ROOT = PROJECT / 'FINAL_REVISION_2026_08'

REVIEWER_10_2 = (
    FINAL_ROOT
    / 'reviewer_10_2_clean_split'
)

OUT = (
    FINAL_ROOT
    / 'final_deployment_lock'
)

OUT.mkdir(
    parents=True,
    exist_ok=True
)

TEST_ROOT = (
    PROJECT
    / 'Data_Clean_LeakageControlled_FINAL'
    / 'Test'
)

IMAGE_SIZE = (224, 224)
SEED = 42
N_BENCH = 100
N_WARMUP = 10

assert PROJECT.exists(), f'STOP: Missing project: {PROJECT}'
assert REVIEWER_10_2.exists(), f'STOP: Missing #10.2 outputs: {REVIEWER_10_2}'
assert TEST_ROOT.exists(), f'STOP: Missing final Test folder: {TEST_ROOT}'

print('TensorFlow:', tf.__version__)
print('Python:', platform.python_version())
print('Platform:', platform.platform())
print('Project:', PROJECT)
print('Output:', OUT)
print('Test root:', TEST_ROOT)

print('\nPhysical devices:')
for d in tf.config.list_physical_devices():
    print(' -', d)

print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — AUDIT EXISTING DEPLOYMENT ARTIFACTS
# ============================================================

KEYWORDS = (
    'tflite',
    'latenc',
    'mobile',
    'android',
    'inference'
)

audit_rows = []

for p in PROJECT.rglob('*'):
    try:
        rel = str(p.relative_to(PROJECT))
    except Exception:
        rel = str(p)

    low = rel.lower()

    if not any(k in low for k in KEYWORDS):
        continue

    size_bytes = np.nan

    if p.is_file():
        try:
            size_bytes = p.stat().st_size
        except Exception:
            pass

    audit_rows.append({
        'relative_path': rel,
        'name': p.name,
        'is_dir': p.is_dir(),
        'suffix': p.suffix.lower() if p.is_file() else '',
        'size_bytes': size_bytes
    })

deployment_audit = pd.DataFrame(
    audit_rows,
    columns=[
        'relative_path',
        'name',
        'is_dir',
        'suffix',
        'size_bytes'
    ]
)

deployment_audit.to_csv(
    OUT / 'Existing_Deployment_Artifact_Audit.csv',
    index=False
)

print(
    'Existing deployment/latency-related paths:',
    len(deployment_audit)
)

if len(deployment_audit):
    display(
        deployment_audit.head(300)
    )

print('\n✅ CELL 2 COMPLETE')

In [ ]:
# ============================================================
# CELL 3 — RESOLVE FINAL MOBILE CHECKPOINTS
# ============================================================

def find_checkpoint_dir(
    parent,
    name_terms
):
    candidates = []

    for d in parent.rglob('*'):
        if not d.is_dir():
            continue

        low = d.name.lower()

        if all(
            term.lower() in low
            for term in name_terms
        ):
            candidates.append(d)

    return sorted(
        candidates,
        key=lambda p: (
            len(str(p)),
            str(p)
        )
    )


def resolve_best_model(
    preferred_dirs,
    label
):
    checked = []

    for d in preferred_dirs:

        if not d.exists():
            checked.append(
                str(d)
            )
            continue

        for name in [
            'best.keras',
            'model.keras',
            'final.keras'
        ]:
            p = d / name

            checked.append(
                str(p)
            )

            if p.exists():
                return p

        keras_files = sorted(
            d.glob('*.keras')
        )

        if keras_files:
            return keras_files[0]

    raise RuntimeError(
        f'STOP: Could not resolve {label}. Checked:\n'
        + '\n'.join(checked)
    )


# Exact expected paths first.
frozen_dir_candidates = [
    REVIEWER_10_2 / 'MobileNetV2_frozen'
]

lastblock_dir_candidates = [
    REVIEWER_10_2 / 'MobileNetV2_last_block',
    REVIEWER_10_2 / 'MobileNetV2_lastblock',
    REVIEWER_10_2 / 'MobileNetV2_finetuned_last_block'
]

# Add fuzzy fallbacks only after exact candidates.
frozen_dir_candidates += find_checkpoint_dir(
    REVIEWER_10_2,
    ['mobilenet', 'frozen']
)

lastblock_dir_candidates += find_checkpoint_dir(
    REVIEWER_10_2,
    ['mobilenet', 'last']
)

FROZEN_MODEL = resolve_best_model(
    frozen_dir_candidates,
    'final frozen MobileNetV2'
)

LASTBLOCK_MODEL = resolve_best_model(
    lastblock_dir_candidates,
    'final last-block MobileNetV2'
)

print('Frozen model:')
print(FROZEN_MODEL)

print('\nLast-block model:')
print(LASTBLOCK_MODEL)

assert FROZEN_MODEL != LASTBLOCK_MODEL

resolved_models = pd.DataFrame([
    {
        'Condition': 'Frozen',
        'keras_path': str(FROZEN_MODEL),
        'keras_size_bytes': FROZEN_MODEL.stat().st_size
    },
    {
        'Condition': 'LastBlock',
        'keras_path': str(LASTBLOCK_MODEL),
        'keras_size_bytes': LASTBLOCK_MODEL.stat().st_size
    }
])

resolved_models.to_csv(
    OUT / 'Resolved_Final_MobileNet_Checkpoints.csv',
    index=False
)

display(resolved_models)

print('\n✅ CELL 3 COMPLETE')

In [ ]:
# ============================================================
# CELL 4 — LOAD LOCKED ACCURACY RESULTS FROM #10.2
# ============================================================

# Locked values from the completed reviewer #10.2 experiment.
# These are not recomputed or retrained here.
accuracy_rows = [
    {
        'Condition': 'Frozen',
        'ThreeClass_Accuracy': 0.993042,
        'Clinical_Accuracy': 0.990715,
        'Clinical_Sensitivity': 0.990632,
        'Clinical_Specificity': 0.990788,
        'Clinical_AUC': 0.999111
    },
    {
        'Condition': 'LastBlock',
        'ThreeClass_Accuracy': 0.996908,
        'Clinical_Accuracy': 0.995631,
        'Clinical_Sensitivity': 0.994145,
        'Clinical_Specificity': 0.996929,
        'Clinical_AUC': 0.9999
    }
]

accuracy_df = pd.DataFrame(
    accuracy_rows
)

accuracy_df.to_csv(
    OUT / 'Locked_MobileNet_Accuracy_Reference.csv',
    index=False
)

display(accuracy_df)

print(
    '\nNote: these values are only reference metrics from the already completed '
    '#10.2 experiment. No training occurs here.'
)

print('\n✅ CELL 4 COMPLETE')

In [ ]:
# ============================================================
# CELL 5 — EXPORT BOTH FINAL MOBILE CHECKPOINTS TO FLOAT32 TFLITE
# ============================================================

TFLITE_DIR = OUT / 'tflite'
TFLITE_DIR.mkdir(parents=True, exist_ok=True)

export_rows = []

for condition, model_path in [
    ('Frozen', FROZEN_MODEL),
    ('LastBlock', LASTBLOCK_MODEL)
]:

    print(
        f'\nLoading {condition}:',
        model_path
    )

    model = tf.keras.models.load_model(
        model_path,
        compile=False
    )

    converter = tf.lite.TFLiteConverter.from_keras_model(
        model
    )

    # Standard float32 export.
    converter.optimizations = []

    tflite_model = converter.convert()

    tflite_path = (
        TFLITE_DIR
        / f'MobileNetV2_{condition}_float32.tflite'
    )

    tflite_path.write_bytes(
        tflite_model
    )

    export_rows.append({
        'Condition': condition,
        'keras_path': str(model_path),
        'keras_size_bytes': model_path.stat().st_size,
        'keras_size_mb': model_path.stat().st_size / (1024**2),
        'tflite_path': str(tflite_path),
        'tflite_size_bytes': tflite_path.stat().st_size,
        'tflite_size_mb': tflite_path.stat().st_size / (1024**2),
        'tflite_to_keras_size_ratio': (
            tflite_path.stat().st_size
            / model_path.stat().st_size
        )
    })

export_df = pd.DataFrame(
    export_rows
)

export_df.to_csv(
    OUT / 'Final_TFLite_Export_Sizes.csv',
    index=False
)

display(export_df)

print('\n✅ CELL 5 COMPLETE')

In [ ]:
# ============================================================
# CELL 6 — AUDIT TFLITE INPUT / OUTPUT TENSORS
# ============================================================

tensor_rows = []

for _, row in export_df.iterrows():

    interpreter = tf.lite.Interpreter(
        model_path=row['tflite_path']
    )

    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    print(
        '\n========================================'
    )
    print(
        row['Condition']
    )
    print(
        '========================================'
    )
    print(
        'Input:',
        input_details
    )
    print(
        'Output:',
        output_details
    )

    if len(input_details) != 1:
        raise RuntimeError(
            f'STOP: {row["Condition"]} TFLite has '
            f'{len(input_details)} inputs, expected 1.'
        )

    if len(output_details) != 1:
        raise RuntimeError(
            f'STOP: {row["Condition"]} TFLite has '
            f'{len(output_details)} outputs, expected 1.'
        )

    in_shape = tuple(
        input_details[0]['shape'].tolist()
    )

    out_shape = tuple(
        output_details[0]['shape'].tolist()
    )

    if in_shape != (1, 224, 224, 3):
        print(
            '⚠️ Input shape differs from expected:',
            in_shape
        )

    tensor_rows.append({
        'Condition': row['Condition'],
        'Input_Name': input_details[0]['name'],
        'Input_Shape': str(in_shape),
        'Input_Dtype': str(input_details[0]['dtype']),
        'Output_Name': output_details[0]['name'],
        'Output_Shape': str(out_shape),
        'Output_Dtype': str(output_details[0]['dtype'])
    })

tensor_df = pd.DataFrame(
    tensor_rows
)

tensor_df.to_csv(
    OUT / 'TFLite_Tensor_Specifications.csv',
    index=False
)

display(tensor_df)

print('\n✅ CELL 6 COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — BUILD CONTROLLED 100-IMAGE BENCHMARK SET
# ============================================================

class_dirs = {
    'Cataract': TEST_ROOT / 'Cataract',
    'Normal': TEST_ROOT / 'Normal',
    'Not Eye': TEST_ROOT / 'Not Eye'
}

for cls, d in class_dirs.items():
    assert d.exists(), f'STOP: Missing class folder: {d}'

IMAGE_SUFFIXES = {
    '.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'
}

class_files = {}

for cls, d in class_dirs.items():

    files = sorted([
        p for p in d.rglob('*')
        if p.is_file()
        and p.suffix.lower() in IMAGE_SUFFIXES
    ])

    class_files[cls] = files

    print(
        cls,
        'files:',
        len(files)
    )

# Balanced-ish 100-image controlled benchmark:
# Cataract 34, Normal 33, Not Eye 33.
rng = np.random.default_rng(
    SEED
)

selection_plan = {
    'Cataract': 34,
    'Normal': 33,
    'Not Eye': 33
}

selected_rows = []

for cls, n in selection_plan.items():

    files = class_files[cls]

    if len(files) < n:
        raise RuntimeError(
            f'STOP: Not enough {cls} test files.'
        )

    idx = rng.choice(
        len(files),
        size=n,
        replace=False
    )

    for i in idx:
        selected_rows.append({
            'class': cls,
            'filepath': str(files[i])
        })

bench_index = pd.DataFrame(
    selected_rows
)

assert len(bench_index) == 100

bench_index.to_csv(
    OUT / 'Deployment_Latency_100_Image_Index.csv',
    index=False
)

display(
    bench_index['class']
    .value_counts()
    .rename_axis('Class')
    .reset_index(name='Images')
)

print('\n✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — PRELOAD / PREPROCESS 100 TEST IMAGES
# ============================================================

images = []

for fp in bench_index['filepath']:

    img = tf.keras.utils.load_img(
        fp,
        target_size=IMAGE_SIZE,
        color_mode='rgb',
        interpolation='nearest'
    )

    arr = tf.keras.utils.img_to_array(
        img
    ).astype(
        np.float32
    )

    arr = arr / 255.0

    images.append(
        arr
    )

images = np.stack(
    images,
    axis=0
)

assert images.shape == (
    100,
    224,
    224,
    3
)

assert images.dtype == np.float32

print('Preloaded images:', images.shape)
print('dtype:', images.dtype)
print('min/max:', float(images.min()), float(images.max()))

np.save(
    OUT / 'Deployment_Latency_100_Preprocessed.npy',
    images
)

print(
    '\nTiming protocol will exclude disk I/O, decode, resize, and normalization.'
)

print('\n✅ CELL 8 COMPLETE')

In [ ]:
# ============================================================
# CELL 9 — KERAS ↔ TFLITE PREDICTION PARITY CHECK
# ============================================================

PARITY_N = 30
PARITY_TOLERANCE = 1e-4

parity_rows = []

for condition, keras_path, tflite_path in [
    (
        'Frozen',
        FROZEN_MODEL,
        TFLITE_DIR / 'MobileNetV2_Frozen_float32.tflite'
    ),
    (
        'LastBlock',
        LASTBLOCK_MODEL,
        TFLITE_DIR / 'MobileNetV2_LastBlock_float32.tflite'
    )
]:

    keras_model = tf.keras.models.load_model(
        keras_path,
        compile=False
    )

    interpreter = tf.lite.Interpreter(
        model_path=str(tflite_path)
    )

    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    max_abs_diffs = []
    class_matches = []

    for i in range(PARITY_N):

        x = images[i:i+1]

        keras_p = keras_model(
            x,
            training=False
        ).numpy()[0]

        interpreter.set_tensor(
            input_detail['index'],
            x.astype(
                input_detail['dtype']
            )
        )

        interpreter.invoke()

        tflite_p = interpreter.get_tensor(
            output_detail['index']
        )[0]

        max_abs_diffs.append(
            float(
                np.max(
                    np.abs(
                        keras_p
                        - tflite_p
                    )
                )
            )
        )

        class_matches.append(
            int(
                np.argmax(keras_p)
                ==
                np.argmax(tflite_p)
            )
        )

    max_diff = max(
        max_abs_diffs
    )

    agreement = np.mean(
        class_matches
    )

    parity_rows.append({
        'Condition': condition,
        'Parity_Images': PARITY_N,
        'Max_Absolute_Probability_Difference': max_diff,
        'Class_Agreement': agreement
    })

    print(
        condition,
        'max abs diff:',
        max_diff,
        'class agreement:',
        agreement
    )

    if agreement < 1.0:
        raise RuntimeError(
            f'STOP: {condition} Keras/TFLite class disagreement detected.'
        )

    if max_diff > PARITY_TOLERANCE:
        print(
            f'⚠️ {condition} max probability difference exceeds '
            f'{PARITY_TOLERANCE}. Inspect before deployment.'
        )

parity_df = pd.DataFrame(
    parity_rows
)

parity_df.to_csv(
    OUT / 'Keras_TFLite_Parity_Audit.csv',
    index=False
)

display(parity_df)

print('\n✅ CELL 9 COMPLETE')

In [ ]:
# ============================================================
# CELL 10 — CONTROLLED TFLITE LATENCY BENCHMARK
# ============================================================

def benchmark_tflite(
    model_path,
    images,
    warmups=10
):

    interpreter = tf.lite.Interpreter(
        model_path=str(model_path),
        num_threads=1
    )

    interpreter.allocate_tensors()

    input_detail = interpreter.get_input_details()[0]
    output_detail = interpreter.get_output_details()[0]

    # Warm-up using real preprocessed images.
    for i in range(warmups):

        x = images[
            i % len(images):
            i % len(images) + 1
        ].astype(
            input_detail['dtype']
        )

        interpreter.set_tensor(
            input_detail['index'],
            x
        )

        interpreter.invoke()

        _ = interpreter.get_tensor(
            output_detail['index']
        )

    times_ms = []

    for i in range(len(images)):

        x = images[
            i:i+1
        ].astype(
            input_detail['dtype']
        )

        interpreter.set_tensor(
            input_detail['index'],
            x
        )

        t0 = time.perf_counter()

        interpreter.invoke()

        # Force output retrieval before stopping the timer.
        _ = interpreter.get_tensor(
            output_detail['index']
        )

        t1 = time.perf_counter()

        times_ms.append(
            (t1 - t0) * 1000.0
        )

    arr = np.asarray(
        times_ms,
        dtype=float
    )

    return arr


latency_rows = []
all_latency = {}

for condition, tflite_path in [
    (
        'Frozen',
        TFLITE_DIR / 'MobileNetV2_Frozen_float32.tflite'
    ),
    (
        'LastBlock',
        TFLITE_DIR / 'MobileNetV2_LastBlock_float32.tflite'
    )
]:

    print(
        '\nBenchmarking:',
        condition
    )

    arr = benchmark_tflite(
        tflite_path,
        images,
        warmups=N_WARMUP
    )

    all_latency[
        condition
    ] = arr

    np.save(
        OUT / f'{condition}_TFLite_Latency_ms.npy',
        arr
    )

    row = {
        'Condition': condition,
        'Warmups': N_WARMUP,
        'Timed_Images': len(arr),
        'Threads': 1,
        'Mean_ms': float(np.mean(arr)),
        'SD_ms': float(np.std(arr, ddof=1)),
        'Median_ms': float(np.median(arr)),
        'P95_ms': float(np.percentile(arr, 95)),
        'Min_ms': float(np.min(arr)),
        'Max_ms': float(np.max(arr)),
        'Throughput_images_per_second': float(
            1000.0 / np.mean(arr)
        )
    }

    latency_rows.append(
        row
    )

latency_df = pd.DataFrame(
    latency_rows
)

latency_df.to_csv(
    OUT / 'Final_TFLite_Controlled_Latency.csv',
    index=False
)

display(latency_df)

print(
    '\nProtocol: model loaded once; 10 warmups; 100 distinct real test images; '
    '224x224 RGB /255 preloaded; disk I/O/decode/resize/normalization excluded; '
    '1 TFLite CPU thread; perf_counter around invoke + output retrieval.'
)

print('\n✅ CELL 10 COMPLETE')

In [ ]:
# ============================================================
# CELL 11 — OPTIONAL KERAS CONTROLLED LATENCY REFERENCE
# ============================================================

@tf.function(
    reduce_retracing=True
)
def infer_keras(
    model,
    x
):
    return model(
        x,
        training=False
    )


keras_latency_rows = []

for condition, keras_path in [
    ('Frozen', FROZEN_MODEL),
    ('LastBlock', LASTBLOCK_MODEL)
]:

    print(
        '\nBenchmarking Keras reference:',
        condition
    )

    model = tf.keras.models.load_model(
        keras_path,
        compile=False
    )

    # Warmup
    for i in range(N_WARMUP):
        _ = model(
            images[i:i+1],
            training=False
        ).numpy()

    times_ms = []

    for i in range(len(images)):

        x = images[i:i+1]

        t0 = time.perf_counter()

        y = model(
            x,
            training=False
        )

        _ = y.numpy()

        t1 = time.perf_counter()

        times_ms.append(
            (t1 - t0) * 1000.0
        )

    arr = np.asarray(
        times_ms,
        dtype=float
    )

    keras_latency_rows.append({
        'Condition': condition,
        'Warmups': N_WARMUP,
        'Timed_Images': len(arr),
        'Mean_ms': float(np.mean(arr)),
        'SD_ms': float(np.std(arr, ddof=1)),
        'Median_ms': float(np.median(arr)),
        'P95_ms': float(np.percentile(arr, 95)),
        'Min_ms': float(np.min(arr)),
        'Max_ms': float(np.max(arr))
    })

keras_latency_df = pd.DataFrame(
    keras_latency_rows
)

keras_latency_df.to_csv(
    OUT / 'Keras_Controlled_Latency_Reference.csv',
    index=False
)

display(
    keras_latency_df
)

print(
    '\nThis Keras timing is a runtime reference only. '
    'The deployment result should use the TFLite timing from Cell 10.'
)

print('\n✅ CELL 11 COMPLETE')

In [ ]:
# ============================================================
# CELL 12 — FINAL DEPLOYMENT COMPARISON
# ============================================================

comparison = (
    accuracy_df
    .merge(
        export_df[
            [
                'Condition',
                'keras_size_mb',
                'tflite_size_mb'
            ]
        ],
        on='Condition',
        how='left'
    )
    .merge(
        latency_df[
            [
                'Condition',
                'Mean_ms',
                'SD_ms',
                'Median_ms',
                'P95_ms',
                'Throughput_images_per_second'
            ]
        ],
        on='Condition',
        how='left'
    )
    .merge(
        parity_df[
            [
                'Condition',
                'Max_Absolute_Probability_Difference',
                'Class_Agreement'
            ]
        ],
        on='Condition',
        how='left'
    )
)

comparison[
    'ThreeClass_Accuracy_pct'
] = 100 * comparison[
    'ThreeClass_Accuracy'
]

comparison[
    'Clinical_Accuracy_pct'
] = 100 * comparison[
    'Clinical_Accuracy'
]

comparison.to_csv(
    OUT / 'FINAL_MobileNet_Deployment_Comparison.csv',
    index=False
)

display(comparison)

print('\n✅ CELL 12 COMPLETE')

In [ ]:
# ============================================================
# CELL 13 — AUTOMATIC DEPLOYMENT RECOMMENDATION
# ============================================================

frozen = comparison[
    comparison['Condition'] == 'Frozen'
].iloc[0]

last = comparison[
    comparison['Condition'] == 'LastBlock'
].iloc[0]

acc_gain_pp = (
    last['Clinical_Accuracy']
    - frozen['Clinical_Accuracy']
) * 100.0

latency_delta_pct = (
    (
        last['Mean_ms']
        - frozen['Mean_ms']
    )
    / frozen['Mean_ms']
    * 100.0
)

size_delta_pct = (
    (
        last['tflite_size_mb']
        - frozen['tflite_size_mb']
    )
    / frozen['tflite_size_mb']
    * 100.0
)

print(
    'Clinical accuracy gain, LastBlock vs Frozen:',
    f'{acc_gain_pp:.4f} percentage points'
)

print(
    'TFLite mean latency delta:',
    f'{latency_delta_pct:.2f}%'
)

print(
    'TFLite size delta:',
    f'{size_delta_pct:.2f}%'
)

# Recommendation:
# Fine-tuning changes weights/trainability but not the MobileNetV2 architecture,
# so size/latency should normally be nearly identical.
# Prefer LastBlock if it keeps parity and has no material deployment penalty.

material_latency_penalty = (
    latency_delta_pct > 10.0
)

material_size_penalty = (
    size_delta_pct > 5.0
)

parity_ok = (
    last['Class_Agreement'] == 1.0
)

if (
    acc_gain_pp > 0
    and not material_latency_penalty
    and not material_size_penalty
    and parity_ok
):
    recommended = 'LastBlock'
    reason = (
        'The last-block fine-tuned MobileNetV2 provides higher locked '
        'clinical and three-class accuracy while retaining the same '
        'MobileNetV2 deployment architecture, with no material TFLite '
        'size or controlled-latency penalty.'
    )
else:
    recommended = 'Frozen'
    reason = (
        'The frozen MobileNetV2 is retained because the last-block '
        'checkpoint does not provide a sufficiently clean deployment '
        'advantage after considering accuracy, size, latency, and parity.'
    )

recommendation = {
    'Recommended_Deployment_Checkpoint':
        recommended,
    'Clinical_Accuracy_Gain_pp_LastBlock_vs_Frozen':
        float(acc_gain_pp),
    'Latency_Delta_pct_LastBlock_vs_Frozen':
        float(latency_delta_pct),
    'TFLite_Size_Delta_pct_LastBlock_vs_Frozen':
        float(size_delta_pct),
    'Reason':
        reason
}

with open(
    OUT / 'FINAL_Deployment_Recommendation.json',
    'w'
) as f:
    json.dump(
        recommendation,
        f,
        indent=2
    )

(
    OUT
    / 'FINAL_Deployment_Recommendation.txt'
).write_text(
    'Recommended deployment checkpoint: '
    + recommended
    + '\n\n'
    + reason
    + '\n'
)

print(
    '\n========================================'
)

print(
    'FINAL DEPLOYMENT RECOMMENDATION'
)

print(
    '========================================'
)

print(
    'Recommended checkpoint:',
    recommended
)

print(
    reason
)

print(
    '\nIMPORTANT: The fair five-model benchmark remains the frozen-baseline '
    'comparison regardless of which MobileNetV2 checkpoint is selected '
    'for deployment.'
)

print('\n✅ CELL 13 COMPLETE')

In [ ]:
# ============================================================
# CELL 14 — MANUSCRIPT-READY DEPLOYMENT METHODS / RESULTS DRAFT
# ============================================================

rec = recommendation[
    'Recommended_Deployment_Checkpoint'
]

rec_row = comparison[
    comparison[
        'Condition'
    ] == rec
].iloc[0]

methods_text = (
    'For deployment evaluation, the selected MobileNetV2 checkpoint was '
    'converted to TensorFlow Lite using the standard float32 converter. '
    'Inference latency was measured after model loading using 100 distinct '
    'held-out test images preloaded and normalized to 224×224 RGB in [0,1]. '
    'Ten warm-up inferences preceded timing. Disk I/O, image decoding, '
    'resizing, and normalization were excluded from the timed region. '
    'Each TFLite model used one CPU thread, and latency was measured with '
    'time.perf_counter() around interpreter.invoke() and output retrieval. '
    'Mean, standard deviation, median, and 95th-percentile latency were reported.'
)

results_text = (
    f'The {rec} MobileNetV2 checkpoint was selected for deployment. '
    f'Its float32 TFLite file size was {rec_row["tflite_size_mb"]:.2f} MB. '
    f'Controlled TFLite inference latency was '
    f'{rec_row["Mean_ms"]:.2f} ± {rec_row["SD_ms"]:.2f} ms '
    f'(median {rec_row["Median_ms"]:.2f} ms; '
    f'P95 {rec_row["P95_ms"]:.2f} ms) under the specified runtime protocol. '
    f'Keras-to-TFLite class agreement was '
    f'{100*rec_row["Class_Agreement"]:.1f}% on the parity audit subset.'
)

(
    OUT
    / 'Deployment_Methods_Draft.txt'
).write_text(
    methods_text
)

(
    OUT
    / 'Deployment_Results_Draft.txt'
).write_text(
    results_text
)

print(
    'METHODS DRAFT:\n'
)

print(
    methods_text
)

print(
    '\nRESULTS DRAFT:\n'
)

print(
    results_text
)

print('\n✅ CELL 14 COMPLETE')

In [ ]:
# ============================================================
# CELL 15 — ANDROID / PHONE RESULT STATUS CHECK
# ============================================================

android_rows = []

for p in PROJECT.rglob('*'):

    if not p.is_file():
        continue

    low = str(p).lower()

    if not any(
        term in low
        for term in [
            'android',
            'phone',
            'latenc'
        ]
    ):
        continue

    android_rows.append({
        'path': str(p),
        'name': p.name,
        'suffix': p.suffix.lower(),
        'size_bytes': p.stat().st_size
    })

android_audit = pd.DataFrame(
    android_rows
)

android_audit.to_csv(
    OUT / 'Android_Phone_Artifact_Audit.csv',
    index=False
)

print(
    'Phone/Android/latency artifacts found:',
    len(android_audit)
)

if len(android_audit):
    display(
        android_audit.head(200)
    )

print(
    '\nNOTE: Colab cannot reproduce the real Android-phone timing. '
    'This audit only identifies existing phone artifacts. '
    'The final phone number must come from the actual physical device '
    'running the final selected TFLite checkpoint.'
)

print('\n✅ CELL 15 COMPLETE')

In [ ]:
# ============================================================
# CELL 16 — FINAL COMPLETION CHECK
# ============================================================

required = [
    'Existing_Deployment_Artifact_Audit.csv',
    'Resolved_Final_MobileNet_Checkpoints.csv',
    'Final_TFLite_Export_Sizes.csv',
    'TFLite_Tensor_Specifications.csv',
    'Deployment_Latency_100_Image_Index.csv',
    'Keras_TFLite_Parity_Audit.csv',
    'Final_TFLite_Controlled_Latency.csv',
    'FINAL_MobileNet_Deployment_Comparison.csv',
    'FINAL_Deployment_Recommendation.json',
    'Deployment_Methods_Draft.txt',
    'Deployment_Results_Draft.txt',
    'Android_Phone_Artifact_Audit.csv'
]

missing = [
    fn
    for fn in required
    if not (
        OUT / fn
    ).exists()
]

if missing:

    print(
        'Missing outputs:'
    )

    for fn in missing:
        print(
            '❌',
            fn
        )

    raise RuntimeError(
        'STOP: Final deployment audit is incomplete.'
    )

(
    OUT
    / 'FINAL_DEPLOYMENT_LOCK_DONE.txt'
).write_text(
    'Final TFLite export and controlled deployment latency audit completed.\n'
    'No training performed.\n'
    f'Recommended checkpoint={recommendation["Recommended_Deployment_Checkpoint"]}\n'
)

print(
    '========================================'
)

print(
    '✅ FINAL TFLITE + DEPLOYMENT LATENCY AUDIT COMPLETE'
)

print(
    '========================================'
)

print(
    '\nSaved to:'
)

print(
    OUT
)

print(
    '\nNEXT: Upload this executed notebook to ChatGPT.'
)

print(
    '\nDo not modify the manuscript deployment numbers until this '
    'executed notebook is inspected.'
)